Seed the map with the MuSIC, add proteins to existing assemblies when both
embeddings support the assignment, find new assemblies among the
remaining proteins and attach them to the closest existing node.

Per-assembly specificity is `spec_raw = (s1 - s2) / (|s1| + |s2|)`,
with `s1` and `s2` the mean intra-assembly cosine similarity in the
HEK293T and HCT116 embeddings. Positive is HEK293T-specific,
negative HCT116-specific.

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
import networkx as nx

from dataclasses import dataclass, asdict
from typing import Dict, Set, List, Optional

from scipy.stats import norm, hypergeom
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AgglomerativeClustering

from statsmodels.stats.multitest import multipletests

In [ ]:
@dataclass
class ExpansionParams:
    # Core
    fdr_alpha: float = 0.05
    n_perm: int = 200
    random_state: int = 42
    margin_min: float = 0.02
    max_add_per_node_ratio: float = 0.25
    sim_threshold_newnode: float = 0.50
    min_newnode_size: int = 3
    parent_attach_min_sim: float = 0.15
    re_iterations: int = 1
    require_parent_for_newnode: bool = True

    # Ontology
    use_ontology: bool = True
    go_alpha: float = 0.05
    go_min_gene_support: int = 1
    go_min_terms_per_node: int = 1
    go_min_terms_per_newnode: int = 1
    # new and middle nodes only: q cutoff for their enriched terms and
    # minimum fraction of their genes annotated to one such term
    go_alpha_newnode: float = 1e-5
    go_min_coverage_newnode: float = 0.75
    go_universe_intersection: bool = True

    # Dual-embedding settings
    use_dual_embeddings: bool = True
    cluster_overlap_jaccard_min: float = 0.3
    parent_sim_aggregate: str = "mean"

    # Middle-node grouping
    # trigger grouping if a parent has >= this many new children
    middle_min_children: int = 4
    # Jaccard threshold to merge child new-nodes
    middle_group_jaccard_min: float = 0.20
    # only create middle node if group has >= this many child nodes
    middle_min_group_size: int = 2



In [ ]:
def _l2_normalize_rows(
    X: np.ndarray,
):
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    return X / norms


def _align_embeddings(
    emb1: pd.DataFrame,
    emb2: pd.DataFrame,
):
    """
    Restrict both embeddings to their common index, sort and
    L2-normalize the rows.
    """
    emb1 = emb1.copy()
    emb1 = emb1.loc[~emb1.index.duplicated(keep="first")]
    emb1 = emb1.sort_index()

    emb2 = emb2.copy()
    emb2 = emb2.loc[~emb2.index.duplicated(keep="first")]
    emb2 = emb2.sort_index()

    common = emb1.index.intersection(emb2.index)
    emb1 = emb1.loc[common]
    emb2 = emb2.loc[common]
    emb1[:] = _l2_normalize_rows(emb1.to_numpy())
    emb2[:] = _l2_normalize_rows(emb2.to_numpy())

    return emb1, emb2


def _build_gene2terms(
    ontology: Dict[str, Set[str]],
    universe: Set[str],
):
    gene2terms: Dict[str, Set[str]] = {}
    for term, gset in ontology.items():
        for g in gset:
            if g not in universe:
                continue
            gene2terms.setdefault(g, set()).add(term)
    return gene2terms


def _bh_fdr(
    pvals: np.ndarray,
    alpha: float,
):
    if pvals.size == 0:
        return np.array([]), np.array([], dtype=bool)
    reject, qvals, _, _ = multipletests(
        pvals,
        alpha=alpha,
        method="fdr_bh",
    )
    return qvals, reject


_ENRICH_COLS = ["term", "k", "n", "N", "M", "pval", "qval"]


def _enrich_terms(
    gene_set: Set[str],
    ontology: Dict[str, Set[str]],
    universe: Set[str],
    alpha: float,
):
    """
    Hypergeometric enrichment of every ontology term in gene_set
    against the universe. Returns the significant terms and the
    full table.
    """
    M = len(universe)
    if M == 0 or len(gene_set) == 0:
        return set(), pd.DataFrame(columns=_ENRICH_COLS)
    gene_set = set(g for g in gene_set if g in universe)
    rows = []
    N = len(gene_set)
    for term, term_genes in ontology.items():
        Tg = set(g for g in term_genes if g in universe)
        n = len(Tg)
        if n == 0:
            continue
        k = len(gene_set & Tg)
        pval = hypergeom.sf(k - 1, M, n, N)
        rows.append((term, k, n, N, M, float(pval)))
    if not rows:
        return set(), pd.DataFrame(columns=_ENRICH_COLS)
    df = pd.DataFrame(rows, columns=_ENRICH_COLS[:-1])
    qval, _ = _bh_fdr(df["pval"].to_numpy(), alpha=alpha)
    df["qval"] = qval
    sig_terms = set(df.loc[df["qval"] <= alpha, "term"].tolist())
    return sig_terms, df.sort_values("qval")


def _seed_centroid(
    emb_df: pd.DataFrame,
    seeds: Set[str],
):
    seeds = [g for g in seeds if g in emb_df.index]
    if len(seeds) == 0:
        raise ValueError(
            "Seed set is empty after intersecting with embedding index."
        )
    vec = emb_df.loc[seeds].to_numpy().mean(axis=0, keepdims=True)
    return _l2_normalize_rows(vec)[0]


def _score_against_centroid(
    emb_df: pd.DataFrame,
    centroid: np.ndarray,
):
    return emb_df.to_numpy() @ centroid


def _null_distribution_by_random_centroids(
    emb_df: pd.DataFrame,
    all_genes: List[str],
    seed_size: int,
    n_perm: int,
    rng: np.random.Generator,
):
    X = emb_df.to_numpy()
    sims = []
    for _ in range(n_perm):
        rand_genes = rng.choice(
            all_genes,
            size=seed_size,
            replace=False,
        )
        centroid = _l2_normalize_rows(
            emb_df.loc[rand_genes].to_numpy().mean(
                axis=0,
                keepdims=True,
            )
        )[0]
        sims.append(X @ centroid)
    sims = np.concatenate(sims, axis=0)
    return float(np.mean(sims)), float(np.std(sims) + 1e-9)


def _per_node_scores_and_stats(
    emb_df: pd.DataFrame,
    node_to_genes: Dict[str, Set[str]],
    fdr_alpha: float,
    n_perm: int,
    rng: np.random.Generator,
):
    """
    For each node, score every gene by cosine similarity to the
    centroid of the node's current members and test it against a
    null of random centroids of the same size.
    """
    all_genes = emb_df.index.to_list()
    rows = []
    node_stats = {}
    for node, seeds in node_to_genes.items():
        seeds = {g for g in seeds if g in emb_df.index}
        if len(seeds) == 0:
            continue
        centroid = _seed_centroid(emb_df, seeds)
        score = _score_against_centroid(emb_df, centroid)
        mu0, sd0 = _null_distribution_by_random_centroids(
            emb_df,
            all_genes,
            seed_size=len(seeds),
            n_perm=n_perm,
            rng=rng,
        )
        z = (score - mu0) / sd0
        pval = 1.0 - norm.cdf(z)
        tmp = pd.DataFrame(
            {
                "gene": all_genes,
                "node": node,
                "score": score,
                "z": z,
                "pval": pval,
            }
        )
        qval, _ = _bh_fdr(tmp["pval"].to_numpy(), alpha=fdr_alpha)
        tmp["qval"] = qval
        rows.append(tmp)
        node_stats[node] = {
            "null_mean": mu0,
            "null_std": sd0,
            "seed_size": len(seeds),
        }
    if rows:
        scores_df = pd.concat(rows, ignore_index=True)
    else:
        scores_df = pd.DataFrame(
            columns=["gene", "node", "score", "z", "pval", "qval"]
        )
    return scores_df, node_stats


def _build_tree_maps(
    edge_df: pd.DataFrame,
):
    parents = {}
    children = {}
    nodes = set(edge_df["parent"].astype(str)).union(
        set(edge_df["child"].astype(str))
    )
    for n in nodes:
        parents[n] = []
        children[n] = []
    for _, row in edge_df.iterrows():
        p = str(row["parent"])
        c = str(row["child"])
        parents[c].append(p)
        children[p].append(c)
    roots = [n for n in nodes if len(parents[n]) == 0]
    return parents, children, roots


def _siblings(
    node: str,
    parents: Dict[str, List[str]],
    children: Dict[str, List[str]],
):
    """
    All other children of the node's parents.
    """
    sibs = set()
    for p in parents.get(node, []):
        for c in children.get(p, []):
            if c != node:
                sibs.add(c)
    return sorted(sibs)


def _first_pass_assignments(
    scores_df: pd.DataFrame,
    edge_df: pd.DataFrame,
    fdr_alpha: float,
    margin_min: float,
):
    """
    FDR + sibling margin -> decision bool per (gene, node).
    """
    parents, children, _ = _build_tree_maps(edge_df)
    node_list = sorted(scores_df["node"].unique().tolist())
    S = scores_df.pivot_table(
        index="gene",
        columns="node",
        values="score",
        aggfunc="max",
        fill_value=0.0,
    )
    rows = []
    for node in node_list:
        sibs = _siblings(node, parents, children)
        if len(sibs):
            comp = S[sibs].max(axis=1).to_numpy()
        else:
            comp = np.zeros((S.shape[0],), dtype=float)
        sub = scores_df[scores_df["node"] == node].copy()
        sub["competing_sib_score"] = comp
        sub["margin"] = sub["score"] - sub["competing_sib_score"]
        # accept if significant and better than the best sibling
        sub["decision"] = (
            (sub["qval"] <= fdr_alpha) & (sub["margin"] >= margin_min)
        )
        rows.append(sub)
    return pd.concat(rows, ignore_index=True)


def _combine_dual_decisions(
    dec1: pd.DataFrame,
    dec2: pd.DataFrame,
):
    """
    Inner join on (gene, node); require BOTH embeddings to accept.
    Keep min(score) as the conservative score.
    """
    cols = [
        "gene",
        "node",
        "score",
        "z",
        "pval",
        "qval",
        "margin",
        "decision",
    ]
    keys = ["gene", "node"]
    d1 = dec1[cols].rename(
        columns={c: f"{c}_e1" for c in cols if c not in keys}
    )
    d2 = dec2[cols].rename(
        columns={c: f"{c}_e2" for c in cols if c not in keys}
    )
    m = d1.merge(d2, on=keys, how="inner")
    m["decision"] = m["decision_e1"] & m["decision_e2"]
    m["score"] = np.minimum(m["score_e1"], m["score_e2"])
    return m


def _gene_support_from_terms(
    gene: str,
    gene2terms: Dict[str, Set[str]],
    enriched_terms: Set[str],
):
    t = gene2terms.get(gene, set())
    return len(t & enriched_terms)


def _cap_growth_per_node(
    decisions: pd.DataFrame,
    node_to_genes: Dict[str, Set[str]],
    max_add_ratio: float,
):
    out_rows = []
    for node, grp in decisions.groupby("node", sort=False):
        sub = grp.copy()
        seeds_n = len(node_to_genes.get(node, set()))
        cap = max(1, int(max_add_ratio * max(1, seeds_n)))
        pos = sub[sub["decision"]].sort_values(
            ["margin_e1", "margin_e2", "score"],
            ascending=[False, False, False],
        )
        keep_idx = pos.head(cap).index
        sub.loc[:, "decision"] = False
        sub.loc[keep_idx, "decision"] = True
        out_rows.append(sub)
    return pd.concat(out_rows, axis=0, ignore_index=True)


def _project_hierarchy_monotonic(
    assign_mat: pd.DataFrame,
    edge_df: pd.DataFrame,
):
    """
    For each gene, force the score to a child node to be no larger
    than the score to its parent.
    """
    base = assign_mat.groupby(
        ["gene", "node"],
        as_index=False,
        sort=False,
    ).agg(score=("score", "max"))
    pivot = base.pivot_table(
        index="gene",
        columns="node",
        values="score",
        aggfunc="max",
        fill_value=0.0,
    ).copy()
    changed = True
    passes = 0
    while changed and passes < 8:
        changed = False
        passes += 1
        for _, row in edge_df.iterrows():
            p = str(row["parent"])
            c = str(row["child"])
            if p not in pivot.columns or c not in pivot.columns:
                continue
            before = pivot[c].to_numpy(copy=True)
            pivot[c] = np.minimum(pivot[c], pivot[p])
            if not np.allclose(before, pivot[c].to_numpy()):
                changed = True
    proj = (
        pivot.stack(future_stack=True)
        .rename("score_proj")
        .reset_index()
    )
    out = assign_mat.merge(proj, on=["gene", "node"], how="left")
    out["score_proj"] = out["score_proj"].fillna(out["score"])
    return out


def _apply_assignments(
    decisions: pd.DataFrame,
    node_to_genes: Dict[str, Set[str]],
):
    updated = {k: set(v) for k, v in node_to_genes.items()}
    for node, grp in decisions.groupby("node", sort=False):
        adds = set(grp.loc[grp["decision"], "gene"].tolist())
        if node not in updated:
            updated[node] = set()
        updated[node].update(adds)
    return updated


def _jaccard(
    a: Set[str],
    b: Set[str],
):
    inter = len(a & b)
    union = len(a | b) if (a or b) else 1
    return inter / union


def _cluster_stats(
    emb_df: pd.DataFrame,
    genes: List[str],
):
    if emb_df is None or len(genes) <= 1:
        return {"mean_intra_sim": 0.0, "size": float(len(genes))}
    X = emb_df.loc[genes].to_numpy()
    D = pairwise_distances(X, metric="cosine")
    sims = 1.0 - D[np.triu_indices_from(D, k=1)]
    return {
        "mean_intra_sim": float(np.mean(sims)),
        "size": float(len(genes)),
    }


def _mean_sim(
    emb: pd.DataFrame,
    genes: List[str],
):
    if emb is not None:
        return _cluster_stats(emb, genes)["mean_intra_sim"]
    return 0.0


def _cluster_genes_embedding(
    emb_df: pd.DataFrame,
    genes: List[str],
    sim_threshold_newnode: float,
    min_size: int,
):
    """
    Average-linkage clustering of the genes by cosine distance,
    cut at 1 - sim_threshold_newnode.
    """
    if emb_df is None or len(genes) < min_size:
        return []
    X = emb_df.loc[genes].to_numpy()
    D = pairwise_distances(X, metric="cosine")
    dist_thr = 1.0 - sim_threshold_newnode
    dist_thr = max(0.0, min(1.0, dist_thr))
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=dist_thr,
    )
    labels = clustering.fit_predict(D)
    clusters = []
    for lab in sorted(set(labels)):
        members = [g for g, l in zip(genes, labels) if l == lab]
        clusters.append(members)
    return clusters


def _place_new_node_parent(
    emb_df1: pd.DataFrame,
    emb_df2: pd.DataFrame,
    genes: List[str],
    node_to_genes: Dict[str, Set[str]],
    parent_sim_aggregate: str,
):
    """
    Existing node whose member centroid is most similar to the
    new cluster, aggregated over the two embeddings.
    """
    best_parent = None
    best_sim1 = -1.0
    best_sim2 = -1.0
    best_agg = -1.0
    if not node_to_genes:
        return None, -1.0, -1.0, -1.0

    for node, seeds in node_to_genes.items():
        if len(seeds) == 0:
            continue
        sim1 = -1.0
        sim2 = -1.0
        if emb_df1 is not None:
            c1 = _seed_centroid(emb_df1, seeds)
            sim1 = float(
                _score_against_centroid(emb_df1.loc[genes], c1).mean()
            )
        if emb_df2 is not None:
            c2 = _seed_centroid(emb_df2, seeds)
            sim2 = float(
                _score_against_centroid(emb_df2.loc[genes], c2).mean()
            )

        if parent_sim_aggregate == "mean":
            agg = np.nanmean([sim1, sim2])
        elif parent_sim_aggregate == "min":
            agg = min(sim1, sim2)
        else:
            raise ValueError("parent_sim_aggregate must be mean or min")

        if agg > best_agg:
            best_agg = agg
            best_parent = node
            best_sim1, best_sim2 = sim1, sim2
    return best_parent, best_sim1, best_sim2, best_agg


def _pick_best_go_term(
    table: pd.DataFrame,
    alpha: float,
):
    """
    Most specific (smallest background) significant term.
    """
    if table is None or table.empty:
        return None
    cand = table.copy()
    cand = cand.loc[cand["qval"] <= alpha].copy()
    if cand.empty:
        return None
    cand["score_ic"] = cand.apply(
        lambda r: float(r["k"]) * float(
            np.log((r["M"] + 1.0) / (r["n"] + 1.0))
        ),
        axis=1,
    )
    cand = cand.sort_values("M")
    row = cand.iloc[0]
    return {
        "term": row["term"],
        "k": int(row["k"]),
        "n": int(row["n"]),
        "N": int(row["N"]),
        "M": int(row["M"]),
        "qval": float(row["qval"]),
        "score_ic": float(row["score_ic"]),
    }


def _node_intrasim_specificity(
    emb1: pd.DataFrame,
    emb2: pd.DataFrame,
    genes: Set[str],
):
    """
    Raw specificity in [-1, 1]: (s1 - s2) / (|s1| + |s2| + eps).
    """
    genes = [g for g in genes if (emb1 is None or g in emb1.index)]
    s1 = _cluster_stats(emb1, genes)["mean_intra_sim"]
    s2 = _cluster_stats(emb2, genes)["mean_intra_sim"]
    denom = abs(s1) + abs(s2) + 1e-9
    return {
        "mean_intra_sim_e1": s1,
        "mean_intra_sim_e2": s2,
        "specificity": (s1 - s2) / denom,
    }

In [ ]:
def _new_node_record(
    genes,
    parent,
    stats1,
    stats2,
    sims,
    sig_terms,
    table,
    origin,
    alpha,
):
    sim1, sim2, sim_agg = sims
    return {
        "genes": set(genes),
        "parent": parent,
        "stats": {
            "mean_intra_sim_e1": stats1,
            "mean_intra_sim_e2": stats2,
            "size": len(genes),
            "parent_sim_e1": sim1,
            "parent_sim_e2": sim2,
            "parent_sim_agg": sim_agg,
        },
        "go_terms": list(sig_terms),
        "best_go": _pick_best_go_term(table, alpha),
        "origin": origin,
    }


def expand_hierarchy(
    emb1_df: pd.DataFrame,
    emb2_df: pd.DataFrame,
    edge_df: pd.DataFrame,
    node_to_genes: Dict[str, Set[str]],
    ontology: Dict[str, Set[str]],
    params: ExpansionParams = ExpansionParams(),
):
    #### Input
    emb1_df, emb2_df = _align_embeddings(emb1_df, emb2_df)

    assert set(edge_df.columns) >= {"parent", "child"}, (
        "edge_df must have columns ['parent', 'child']"
    )
    edge_df = edge_df.copy()
    edge_df["parent"] = edge_df["parent"].astype(str)
    edge_df["child"] = edge_df["child"].astype(str)

    rng = np.random.default_rng(params.random_state)

    universe = set(emb1_df.index).intersection(set(emb2_df.index))
    if ontology is None:
        params.use_ontology = False
    use_go = params.use_ontology and ontology is not None
    if use_go and not params.go_universe_intersection:
        universe = set(universe) | set().union(*ontology.values())

    # Significantly enriched GO terms of each seed node
    gene2terms: Dict[str, Set[str]] = {}
    enriched_terms_per_node: Dict[str, Set[str]] = {}
    if use_go:
        gene2terms = _build_gene2terms(ontology, universe)
        for node, seeds in node_to_genes.items():
            seeds_u = set(g for g in seeds if g in universe)
            sig_terms, _ = _enrich_terms(
                seeds_u,
                ontology,
                universe,
                alpha=params.go_alpha,
            )
            if len(sig_terms) >= params.go_min_terms_per_node:
                enriched_terms_per_node[node] = sig_terms
            else:
                enriched_terms_per_node[node] = set()

    def go_gate(row):
        # a gene joins a node only if it carries at least one of
        # the node's enriched terms (nodes without terms accept all)
        if not row["decision"]:
            return False
        sig_terms = enriched_terms_per_node.get(row["node"], set())
        if len(sig_terms) < params.go_min_terms_per_node:
            return True
        support = _gene_support_from_terms(
            row["gene"],
            gene2terms,
            sig_terms,
        )
        return support >= params.go_min_gene_support

    def one_round(current, iteration):
        # score, decide in both embeddings, gate, cap, project
        s1, st1 = _per_node_scores_and_stats(
            emb1_df,
            current,
            params.fdr_alpha,
            params.n_perm,
            rng,
        )
        d1 = _first_pass_assignments(
            s1,
            edge_df,
            params.fdr_alpha,
            params.margin_min,
        )
        s2, st2 = _per_node_scores_and_stats(
            emb2_df,
            current,
            params.fdr_alpha,
            params.n_perm,
            rng,
        )
        d2 = _first_pass_assignments(
            s2,
            edge_df,
            params.fdr_alpha,
            params.margin_min,
        )
        dec = _combine_dual_decisions(d1, d2)
        if use_go:
            dec["decision"] = dec.apply(go_gate, axis=1)
        dec = _cap_growth_per_node(
            dec,
            current,
            params.max_add_per_node_ratio,
        )
        proj = _project_hierarchy_monotonic(
            dec[["gene", "node", "score"]],
            edge_df,
        )
        dec = dec.merge(
            proj[["gene", "node", "score_proj"]],
            on=["gene", "node"],
            how="left",
        )
        dec["score"] = dec["score_proj"].fillna(dec["score"])
        dec.drop(columns=["score_proj"], inplace=True)
        dec["iteration"] = iteration
        return dec, st1, st2

    #### Assign genes to existing nodes, optionally re-iterate
    decisions, stats1, stats2 = one_round(node_to_genes, 0)
    updated_node_to_genes = _apply_assignments(decisions, node_to_genes)
    all_iters = [decisions.copy()]
    for it in range(1, params.re_iterations + 1):
        dec_b, _, _ = one_round(updated_node_to_genes, it)
        updated_node_to_genes = _apply_assignments(
            dec_b,
            updated_node_to_genes,
        )
        all_iters.append(dec_b.copy())

    assign_table = pd.concat(all_iters, ignore_index=True)
    assigned_genes = set(
        assign_table.loc[assign_table["decision"], "gene"].unique()
    )
    if node_to_genes:
        seed_genes = set().union(
            *[set(v) for v in node_to_genes.values()]
        )
    else:
        seed_genes = set()
    already_in = assigned_genes.union(seed_genes)
    remaining = [g for g in emb1_df.index if g not in already_in]

    #### New clusters among the remaining genes
    clusters_e1 = _cluster_genes_embedding(
        emb1_df,
        remaining,
        params.sim_threshold_newnode,
        params.min_newnode_size,
    )
    clusters_e2 = _cluster_genes_embedding(
        emb2_df,
        remaining,
        params.sim_threshold_newnode,
        params.min_newnode_size,
    )
    matched_e2 = set()
    new_nodes = {}
    node_specificity = {}
    dropped_genes = set(remaining)

    def go_check(genes):
        if not use_go:
            return True, set(), None
        sig_terms, table = _enrich_terms(
            set(genes),
            ontology,
            universe,
            alpha=params.go_alpha,
        )
        strong = table[table["qval"] <= params.go_alpha_newnode]
        cover = (strong["k"] / strong["N"]).max() if len(strong) else 0.0
        ok = (
            len(strong) >= params.go_min_terms_per_newnode
            and cover >= params.go_min_coverage_newnode
        )
        return ok, sig_terms, table

    def add_new_node(genes, origin, sims, parent, sig_terms, table):
        node_name = f"newnode_{origin}_{len(new_nodes) + 1:03d}"
        new_nodes[node_name] = _new_node_record(
            genes,
            parent,
            _mean_sim(emb1_df, genes),
            _mean_sim(emb2_df, genes),
            sims,
            sig_terms,
            table,
            origin,
            params.go_alpha,
        )
        dropped_genes.difference_update(genes)
        node_specificity[node_name] = _node_intrasim_specificity(
            emb1_df,
            emb2_df,
            set(genes),
        )

    # shared new nodes: an emb1 cluster that is also tight in emb2
    # or overlaps an emb2 cluster
    for gset1 in clusters_e1:
        s2 = _mean_sim(emb2_df, gset1)
        best_e2_idx = -1
        best_j = 0.0
        for idx2, gset2 in enumerate(clusters_e2):
            if idx2 in matched_e2:
                continue
            j = _jaccard(set(gset1), set(gset2))
            if j > best_j:
                best_j = j
                best_e2_idx = idx2
        is_shared_by_threshold = s2 >= params.sim_threshold_newnode
        is_shared_by_match = (
            best_j >= params.cluster_overlap_jaccard_min
        )
        if not (is_shared_by_threshold or is_shared_by_match):
            continue
        if best_e2_idx >= 0:
            genes = sorted(set(gset1) | set(clusters_e2[best_e2_idx]))
        else:
            genes = gset1
        stats1 = _cluster_stats(emb1_df, genes)
        stats2 = _cluster_stats(emb2_df, genes)
        ok_go, sig_terms, table = go_check(genes)
        tight = (
            stats1["size"] >= params.min_newnode_size
            and stats1["mean_intra_sim"] >= params.sim_threshold_newnode
            and stats2["mean_intra_sim"] >= params.sim_threshold_newnode
        )
        if not (tight and ok_go):
            continue
        parent, sim1, sim2, sim_agg = _place_new_node_parent(
            emb1_df,
            emb2_df,
            genes,
            updated_node_to_genes,
            params.parent_sim_aggregate,
        )
        attach_ok = parent is not None and (
            sim_agg >= params.parent_attach_min_sim
        )
        if not params.require_parent_for_newnode or attach_ok:
            add_new_node(
                genes,
                "shared",
                (sim1, sim2, sim_agg),
                parent,
                sig_terms,
                table,
            )
            if best_e2_idx >= 0:
                matched_e2.add(best_e2_idx)

    # single-embedding new nodes
    def single_embedding_nodes(clusters, emb_df, origin, skip):
        for idx, gset in enumerate(clusters):
            if idx in skip:
                continue
            if all(g not in dropped_genes for g in gset):
                continue
            stats = _cluster_stats(emb_df, gset)
            if stats["size"] < params.min_newnode_size:
                continue
            if stats["mean_intra_sim"] < params.sim_threshold_newnode:
                continue
            ok_go, sig_terms, table = go_check(gset)
            if not ok_go:
                continue
            parent, sim1, sim2, sim_agg = _place_new_node_parent(
                emb1_df,
                emb2_df,
                gset,
                updated_node_to_genes,
                params.parent_sim_aggregate,
            )
            own_sim = sim1 if origin == "e1" else sim2
            attach_ok = parent is not None and (
                own_sim >= params.parent_attach_min_sim
            )
            if not params.require_parent_for_newnode or attach_ok:
                add_new_node(
                    gset,
                    origin,
                    (sim1, sim2, sim_agg),
                    parent,
                    sig_terms,
                    table,
                )

    single_embedding_nodes(clusters_e1, emb1_df, "e1", set())
    single_embedding_nodes(clusters_e2, emb2_df, "e2", matched_e2)

    # raw specificity of the original (expanded) nodes
    for node, genes in updated_node_to_genes.items():
        node_specificity[node] = _node_intrasim_specificity(
            emb1_df,
            emb2_df,
            set(genes),
        )

    #### Middle nodes: group overlapping new children of one parent
    parent_to_newchildren: Dict[str, List[str]] = {}
    for nname, info in new_nodes.items():
        p = info.get("parent")
        if p is None:
            continue
        parent_to_newchildren.setdefault(p, []).append(nname)

    def middle_node_ok(genes):
        genes_list = sorted(genes)
        stats1 = _cluster_stats(emb1_df, genes_list)
        stats2 = _cluster_stats(emb2_df, genes_list)
        embed_ok = (
            stats1["mean_intra_sim"] >= params.sim_threshold_newnode
            or stats2["mean_intra_sim"] >= params.sim_threshold_newnode
        )
        go_ok, go_terms, table_mid = go_check(genes_list)
        best_go = _pick_best_go_term(table_mid, params.go_alpha)
        stats = {
            "mean_intra_sim_e1": stats1["mean_intra_sim"],
            "mean_intra_sim_e2": stats2["mean_intra_sim"],
            "size": len(genes_list),
        }
        return (embed_ok and go_ok), stats, best_go, go_terms

    midnode_counter = 0
    for parent, kids in parent_to_newchildren.items():
        if len(kids) < params.middle_min_children:
            continue
        kid_genes = [new_nodes[k]["genes"] for k in kids]
        n = len(kids)
        J = np.zeros((n, n), dtype=float)
        for i in range(n):
            for j in range(i + 1, n):
                J[i, j] = J[j, i] = _jaccard(kid_genes[i], kid_genes[j])
        clustering = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=1.0 - params.middle_group_jaccard_min,
        )
        labels = clustering.fit_predict(1.0 - J)

        groups: Dict[int, List[int]] = {}
        for idx, lab in enumerate(labels):
            groups.setdefault(lab, []).append(idx)

        for lab, idxs in groups.items():
            if len(idxs) < params.middle_min_group_size:
                continue
            union_genes = set().union(*[kid_genes[i] for i in idxs])
            if len(union_genes) < params.min_newnode_size:
                continue
            ok, stats, best_go, go_terms = middle_node_ok(union_genes)
            if not ok:
                continue
            midnode_counter += 1
            midname = f"midnode_{parent}_{midnode_counter:03d}"
            new_nodes[midname] = {
                "genes": union_genes,
                "parent": parent,
                "stats": {
                    **stats,
                    "parent_sim_e1": None,
                    "parent_sim_e2": None,
                    "parent_sim_agg": None,
                },
                "go_terms": list(go_terms),
                "best_go": best_go,
                "origin": "middle",
                "children": [kids[i] for i in idxs],
            }
            node_specificity[midname] = _node_intrasim_specificity(
                emb1_df,
                emb2_df,
                union_genes,
            )
            for i in idxs:
                new_nodes[kids[i]]["parent"] = midname

    # prune children identical to their middle node
    for _mid, _info in list(new_nodes.items()):
        if _info.get("origin") != "middle":
            continue
        mid_genes = set(_info.get("genes", set()))
        new_children = []
        for ch in list(_info.get("children", [])):
            ch_info = new_nodes.get(ch)
            if ch_info is None:
                continue
            if set(ch_info.get("genes", set())) == mid_genes:
                new_nodes.pop(ch, None)
            else:
                new_children.append(ch)
        _info["children"] = new_children

    diagnostics = {
        "n_assigned_genes": int(len(assigned_genes)),
        "n_remaining_for_new_nodes": int(len(remaining)),
        "node_stats_e1": stats1,
        "node_stats_e2": stats2,
    }

    return {
        "updated_node_to_genes": updated_node_to_genes,
        "assign_table": assign_table.sort_values(
            ["iteration", "node", "decision", "score"],
            ascending=[True, True, False, False],
        ).reset_index(drop=True),
        "new_nodes": new_nodes,
        "dropped_genes": sorted(dropped_genes),
        "params": asdict(params),
        "diagnostics": diagnostics,
        "node_specificity": node_specificity,
    }

In [ ]:
def _build_final_edge_df(
    edge_df: pd.DataFrame,
    new_nodes: Dict[str, dict],
):
    """
    Add parent->new_node (and middle->child) edges to the original
    edge list and dedupe.
    """
    edge_df = edge_df.copy()
    edge_df["parent"] = edge_df["parent"].astype(str)
    edge_df["child"] = edge_df["child"].astype(str)

    extra_edges = []
    for node_name, info in (new_nodes or {}).items():
        parent = info.get("parent", None)
        if parent is None:
            continue
        extra_edges.append(
            {"parent": str(parent), "child": str(node_name)}
        )
        if info.get("origin") == "middle":
            for ch in info.get("children", []):
                extra_edges.append(
                    {"parent": str(node_name), "child": str(ch)}
                )

    if not extra_edges:
        return edge_df.copy()
    add_df = pd.DataFrame(extra_edges)
    return (
        pd.concat([edge_df, add_df], ignore_index=True)
        .drop_duplicates(subset=["parent", "child"])
        .reset_index(drop=True)
    )


def _rollup_node_genes(
    final_edge_df: pd.DataFrame,
    updated_node_to_genes: Dict[str, Set[str]],
    new_nodes: Dict[str, dict],
):
    """
    node -> genes including seeds, new assignments and all
    descendants.
    """
    node_genes: Dict[str, Set[str]] = {
        k: set(v) for k, v in (updated_node_to_genes or {}).items()
    }
    for node_name, info in (new_nodes or {}).items():
        node_genes.setdefault(node_name, set()).update(
            set(info.get("genes", set()))
        )

    all_nodes = pd.unique(
        final_edge_df[["parent", "child"]].values.ravel("K")
    ).astype(str)
    for n in all_nodes:
        node_genes.setdefault(n, set())

    G = nx.DiGraph()
    G.add_edges_from(
        final_edge_df[["parent", "child"]].itertuples(
            index=False,
            name=None,
        )
    )
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(
            "Hierarchy contains a cycle; cannot roll up genes."
        )

    topo = list(nx.topological_sort(G))
    for node in reversed(topo):
        for child in G.successors(node):
            node_genes[node].update(node_genes.get(child, set()))

    return node_genes


def _enrich_and_annotate_all_nodes(
    final_node_to_genes: Dict[str, Set[str]],
    ontology: Dict[str, Set[str]],
    alpha: float = 0.05,
    universe: Optional[Set[str]] = None,
):
    """
    Re-run GO over the final node gene sets and pick a best term
    per node.
    """
    if universe is None:
        if len(ontology):
            ontu = set().union(*ontology.values())
        else:
            ontu = set()
        if len(final_node_to_genes):
            nodeu = set().union(*final_node_to_genes.values())
        else:
            nodeu = set()
        universe = (ontu & nodeu) if len(ontu) else nodeu

    best_go_per_node: Dict[str, Optional[dict]] = {}
    tables_per_node: Dict[str, pd.DataFrame] = {}

    for node, genes in final_node_to_genes.items():
        sig_terms, table = _enrich_terms(
            set(genes),
            ontology,
            universe,
            alpha=alpha,
        )
        if table is None or table.empty:
            best_go = None
            tables_per_node[node] = pd.DataFrame(
                columns=_ENRICH_COLS + ["score_ic"]
            )
        else:
            table = table.copy()
            table["score_ic"] = table.apply(
                lambda r: float(r["k"]) * float(
                    np.log((r["M"] + 1.0) / (r["n"] + 1.0))
                ),
                axis=1,
            )
            tables_per_node[node] = table.sort_values(
                ["qval", "score_ic"],
                ascending=[True, False],
            )
            best_go = _pick_best_go_term(table, alpha)
        best_go_per_node[node] = best_go

    return best_go_per_node, tables_per_node


def finalize_hierarchy(
    edge_df: pd.DataFrame,
    pipeline_output: dict,
    ontology: Optional[Dict[str, Set[str]]] = None,
    go_alpha: float = 0.05,
    universe: Optional[Set[str]] = None,
):
    """
    Post-processing wrapper:
    - adds parent->new_node (and middle->child) edges
    - returns final node->genes (seeds + new + descendants)
    - optionally re-runs GO enrichment on the final gene sets
    """
    updated_node_to_genes = pipeline_output.get(
        "updated_node_to_genes",
        {},
    )
    new_nodes = pipeline_output.get("new_nodes", {})
    final_edge_df = _build_final_edge_df(edge_df, new_nodes)
    final_node_to_genes = _rollup_node_genes(
        final_edge_df,
        updated_node_to_genes,
        new_nodes,
    )

    best_go_per_node = None
    if ontology is not None:
        best_go_per_node, _ = _enrich_and_annotate_all_nodes(
            final_node_to_genes,
            ontology,
            alpha=go_alpha,
            universe=universe,
        )

    return final_edge_df, final_node_to_genes, best_go_per_node

### Specificity score

In [ ]:
def _node_pair_stats(
    emb1: pd.DataFrame,
    emb2: pd.DataFrame,
    genes: Set[str],
):
    """
    Number of embedded genes and mean intra-node pair similarity in
    both embeddings. The embeddings must be aligned.
    """
    genes = sorted(g for g in genes if g in emb1.index)
    n = len(genes)
    out = {"n": n, "s1": np.nan, "s2": np.nan}
    if n < 2:
        return out
    X1 = emb1.loc[genes].to_numpy()
    X2 = emb2.loc[genes].to_numpy()
    iu = np.triu_indices(n, k=1)
    out["s1"] = float((X1 @ X1.T)[iu].mean())
    out["s2"] = float((X2 @ X2.T)[iu].mean())
    return out


def score_specificity(
    emb1: pd.DataFrame,
    emb2: pd.DataFrame,
    node_to_genes: Dict[str, Set[str]],
):
    """
    Per-node raw specificity. Positive means more HEK293T-specific,
    negative more HCT116-specific; nodes with < 2 embedded genes
    get 0.
    """
    rows = []
    for node, genes in node_to_genes.items():
        st = _node_pair_stats(emb1, emb2, genes)
        rows.append({"node": node, **st})
    df = pd.DataFrame(rows).set_index("node")
    s1 = df["s1"]
    s2 = df["s2"]
    df["spec_raw"] = (s1 - s2) / (s1.abs() + s2.abs() + 1e-9)
    df["spec_raw"] = df["spec_raw"].fillna(0.0)
    return df

In [ ]:
import ndex2
import matplotlib.pyplot as plt
from goatools.anno.gaf_reader import GafReader
from goatools.obo_parser import GODag

plt.style.use("/home/jig049/misc/style.mplstyle")

In [ ]:
# read in embedding
emb_df1 = pd.read_csv(
    f"../../result/HEK293T_HCT116/coemb/promergeclub_HCT116anchor/HEK293T/HEK293T_latent.tsv",
    sep="\t",
    index_col=0,
)
emb_df1.columns = range(emb_df1.shape[1])

emb_df2 = pd.read_csv(
    f"../../result/HEK293T_HCT116/coemb/promergeclub_HEK293Tanchor/HCT116/HCT116_latent.tsv",
    sep="\t",
    index_col=0,
)
emb_df2.columns = range(emb_df2.shape[1])

# read in music hierarchy
# output1 edge list in pd.df
# output2 node to genes in dict
net = ndex2.create_nice_cx_from_file(
    "../../data/MuSIC/Multi-Scale Integrated Cell (MuSIC) v1.cx"
)

edge_df = net.to_pandas_dataframe()
edge_df = edge_df.iloc[:, [0, 2]].copy()
edge_df.columns = ["parent", "child"]
edge_df["parent"] = "music_" + edge_df["parent"].astype(str)
edge_df["child"] = "music_" + edge_df["child"].astype(str)

music_node = []
for k in net.nodes.keys():
    music_node.append(
        [
            net.nodes[k]["n"],
            net.nodeAttributes[k][0]["v"],
            net.nodeAttributes[k][3]["v"],
            net.nodeAttributes[k][6]["v"],
        ]
    )

music_node = pd.DataFrame(music_node)
music_node[2] = music_node[2].astype(int)
music_node = music_node.sort_values(2, ascending=False)
music_node = music_node.reset_index(drop=True)
music_node.index = "music_" + music_node[0].astype(str)
music_node_genes = music_node[1].str.split(", ").apply(set)
node_to_genes = music_node_genes.to_dict()

In [ ]:
# parse GO
obodag = GODag("../../data/GO/go-basic.obo")
gaf = GafReader("../../data/GO/goa_human.gaf")

go2genes = {}
go2name = {}
for annotation in gaf.associations:
    if annotation.NS != "CC":
        continue
    go_id = annotation.GO_ID
    go2genes.setdefault(go_id, set()).add(annotation.DB_Symbol)
    go2name[go_id] = obodag[go_id].name

In [ ]:
params = ExpansionParams()

result = expand_hierarchy(
    emb_df1,
    emb_df2,
    edge_df,
    node_to_genes,
    params=params,
    ontology=go2genes,
)

final = finalize_hierarchy(
    edge_df,
    result,
    ontology=go2genes,
    go_alpha=0.05,
)
final_edge_df, final_node_to_genes, best_go_per_node = final

In [ ]:
score_sets = dict(result["updated_node_to_genes"])
for name, info in result["new_nodes"].items():
    score_sets[name] = set(info["genes"])

emb_a1, emb_a2 = _align_embeddings(emb_df1, emb_df2)
score_df = score_specificity(
    emb_a1,
    emb_a2,
    score_sets,
)

In [ ]:
# node table
final_node_df = pd.DataFrame(
    {
        "node": list(final_node_to_genes.keys()),
        "genes": [
            ", ".join(sorted(v)) for v in final_node_to_genes.values()
        ],
        "size": [len(v) for v in final_node_to_genes.values()],
    }
)
final_node_df["log_size"] = np.log(final_node_df["size"])
final_node_df["best_go"] = final_node_df["node"].map(
    {
        k: (v["term"] if v is not None else None)
        for k, v in (best_go_per_node or {}).items()
    }
)
final_node_df["name"] = final_node_df["best_go"].map(go2name).astype(
    object
)
final_node_df["is_new"] = ~final_node_df["node"].str.startswith(
    "music_"
)
seed_names = music_node[3].to_dict()
final_node_df.loc[~final_node_df["is_new"], "name"] = (
    final_node_df.loc[~final_node_df["is_new"], "node"].map(seed_names)
)
final_node_df["origin"] = final_node_df["node"].map(
    {k: v["origin"] for k, v in result["new_nodes"].items()}
).fillna("music")

for col in ["n", "s1", "s2", "spec_raw"]:
    final_node_df[col] = final_node_df["node"].map(score_df[col])
final_node_df = final_node_df.rename(columns={"n": "n_embedded"})

# labels: strongest scores among new nodes plus the largest seeds
final_node_df["label"] = ""
new = final_node_df[final_node_df["is_new"]]
scored = new.sort_values("spec_raw")
label_idx = np.concatenate(
    [
        scored.index[:10].values,
        scored.index[-10:].values,
        final_node_df[~final_node_df["is_new"]]
        .sort_values("size", ascending=False)
        .index[:10]
        .values,
    ]
)
final_node_df.loc[label_idx, "label"] = final_node_df.loc[
    label_idx,
    "name",
]

# extra name columns, appended after the existing ones
final_node_df["music_name"] = (
    final_node_df["node"]
    .map(seed_names)
    .where(~final_node_df["is_new"], "")
)
final_node_df["best_go_name"] = (
    final_node_df["best_go"].map(go2name).astype(object)
)

final_node_df = final_node_df.sort_values(
    ["is_new", "size"],
    ascending=[True, False],
).reset_index(drop=True)

In [ ]:
out_dir = "../../result/HEK293T_HCT116/music"
os.makedirs(out_dir, exist_ok=True)

final_edge_df.to_csv(f"{out_dir}/hier_edge.tsv", sep="\t", index=False)
final_node_df.to_csv(f"{out_dir}/hier_node.tsv", sep="\t", index=False)
pd.Series(asdict(params)).to_csv(
    f"{out_dir}/hier_params.tsv",
    sep="\t",
    header=False,
)